In [1]:
import pandas as pd
import pgserver

from sqlalchemy import create_engine, text

df = pd.read_csv("infectious_cases.csv")

df.columns = df.columns.str.lower()

display(df.head())
print("Розмір датасету:", df.shape)

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


Розмір датасету: (10521, 12)


In [2]:
pg = pgserver.get_server(
    '/tmp/fp_pg',
    cleanup_mode='stop'
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [3]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS pandemic;"))
    conn.execute(text("SET search_path TO pandemic;"))

    print("Поточна схема:",
          conn.execute(text("SELECT current_schema()")).scalar_one())

    df.to_sql(
        "infectious_cases",
        con=conn,
        schema="pandemic",
        if_exists="replace",
        index=False
    )

print("Дані успішно імпортовано!1")

Поточна схема: pandemic
Дані успішно імпортовано!1


In [4]:
with engine.connect() as conn:
    result = pd.read_sql_query(
        text("SELECT * FROM pandemic.infectious_cases LIMIT 10;"),
        conn
    )

display(result)

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160.0,0.0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859.0,0.0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730.0,0.0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937.0,0.0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864.0,0.0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867.0,0.0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901.0,0.0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396.0,0.0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149.0,0.0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385.0,0.0,None,None,None,None,None,None


In [5]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM pandemic.infectious_cases;")
    ).scalar_one()

print("Кількість записів:", count)

Кількість записів: 10521


Завдання 2

In [6]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS pandemic.infectious_cases_normalized;
    """))

    conn.execute(text("""
        DROP TABLE IF EXISTS pandemic.entities;
    """))

print("таблиця очищена")

таблиця очищена


In [7]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS pandemic.entities (
            id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            entity TEXT NOT NULL,
            code TEXT,
            UNIQUE (entity, code)
        );
    """))

    conn.execute(text("""
        INSERT INTO pandemic.entities (entity, code)
        SELECT DISTINCT i.entity, i.code
        FROM pandemic.infectious_cases AS i
        WHERE NOT EXISTS (
            SELECT 1
            FROM pandemic.entities AS e
            WHERE e.entity = i.entity
              AND e.code IS NOT DISTINCT FROM i.code
        )
        ON CONFLICT (entity, code) DO NOTHING;
    """))

print("Таблиця entities-")

Таблиця entities-


In [8]:
query = """
SELECT COUNT(*) AS total_entities
FROM pandemic.entities;
"""

display(pd.read_sql(query, engine))

,total_entities
0,245


In [9]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS pandemic.infectious_cases_normalized (
            id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            entity_id BIGINT NOT NULL REFERENCES pandemic.entities(id),
            year INTEGER NOT NULL,
            number_yaws DOUBLE PRECISION,
            polio_cases DOUBLE PRECISION,
            cases_guinea_worm DOUBLE PRECISION,
            number_rabies DOUBLE PRECISION,
            number_malaria DOUBLE PRECISION,
            number_hiv DOUBLE PRECISION,
            number_tuberculosis DOUBLE PRECISION,
            number_smallpox DOUBLE PRECISION,
            number_cholera_cases DOUBLE PRECISION,
            UNIQUE (entity_id, year)
        );
    """))

print("Таблиця infectious_cases_normalized")

Таблиця infectious_cases_normalized


In [10]:
query = """
SELECT *
FROM pandemic.entities
ORDER BY id
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,id,entity,code
0,1,Morocco,MAR
1,2,Russia,RUS
2,3,Democratic Republic of Congo,COD
3,4,Trinidad and Tobago,TTO
4,5,Antigua and Barbuda,ATG
5,6,Libya,LBY
6,7,Liberia,LBR
7,8,Greece,GRC
8,9,Cuba,CUB
9,10,Nicaragua,NIC


In [11]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO pandemic.infectious_cases_normalized (
            entity_id,
            year,
            number_yaws,
            polio_cases,
            cases_guinea_worm,
            number_rabies,
            number_malaria,
            number_hiv,
            number_tuberculosis,
            number_smallpox,
            number_cholera_cases
        )
        SELECT
            e.id,
            i.year,
            i.number_yaws,
            i.polio_cases,
            i.cases_guinea_worm,
            i.number_rabies,
            i.number_malaria,
            i.number_hiv,
            i.number_tuberculosis,
            i.number_smallpox,
            i.number_cholera_cases
        FROM pandemic.infectious_cases AS i
        JOIN pandemic.entities AS e
            ON i.entity = e.entity
            AND i.code IS NOT DISTINCT FROM e.code
        ON CONFLICT (entity_id, year) DO NOTHING;
    """))

print("infectious_cases_normalized")

infectious_cases_normalized


In [12]:
query = """
SELECT *
FROM pandemic.infectious_cases_normalized
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,189,1990,None,336.0,0.0,2.145772,1219845.80,87.665596,16303.112,None,NaN
1,2,189,1991,None,14.0,0.0,2.320975,1132781.50,103.773590,17582.254,None,NaN
2,3,189,1995,None,0.0,0.0,3.126361,725758.30,143.545270,22163.428,None,19903.0
3,4,189,1997,None,21.0,0.0,3.446122,605853.06,160.595250,23140.863,None,4170.0
4,5,189,1998,None,65.0,0.0,3.557025,547045.40,164.607250,23179.447,None,10000.0
5,6,189,1999,None,167.0,0.0,3.607037,502898.88,168.993590,22944.182,None,24639.0
6,7,189,2000,None,133.0,0.0,3.636235,491614.72,177.666750,22879.678,None,4330.0
7,8,189,2001,None,22.0,0.0,3.711832,511626.20,192.514570,23330.460,None,4499.0
8,9,189,2002,None,11.0,0.0,3.841707,546030.60,208.172960,24148.050,None,3.0
9,10,189,2003,None,9.0,0.0,3.914814,574740.30,220.204760,24519.380,None,41.0


In [13]:
query = """
SELECT
    (SELECT COUNT(*) FROM pandemic.infectious_cases) AS original_rows,
    (SELECT COUNT(*) FROM pandemic.entities) AS entities_count,
    (SELECT COUNT(*) FROM pandemic.infectious_cases_normalized) AS normalized_rows;
"""

display(pd.read_sql(query, engine))

,original_rows,entities_count,normalized_rows
0,10521,245,10521


In [14]:
query = """
SELECT
    e.entity,
    e.code,
    n.year,
    n.polio_cases,
    n.number_malaria,
    n.number_hiv
FROM pandemic.infectious_cases_normalized AS n
JOIN pandemic.entities AS e
    ON n.entity_id = e.id
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,entity,code,year,polio_cases,number_malaria,number_hiv
0,Morocco,MAR,1971,NaN,None,None
1,Morocco,MAR,1972,NaN,None,None
2,Morocco,MAR,1976,NaN,None,None
3,Morocco,MAR,1980,364.0,None,None
4,Morocco,MAR,1981,406.0,None,None
5,Morocco,MAR,1982,623.0,None,None
6,Morocco,MAR,1983,343.0,None,None
7,Morocco,MAR,1984,154.0,None,None
8,Morocco,MAR,1985,105.0,None,None
9,Morocco,MAR,1986,84.0,None,None


Завдання 3

In [15]:
query = """
SELECT
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies)::numeric, 2) AS avg_rabies,
    MIN(n.number_rabies) AS min_rabies,
    MAX(n.number_rabies) AS max_rabies,
    SUM(n.number_rabies) AS sum_rabies
FROM pandemic.infectious_cases_normalized AS n
JOIN pandemic.entities AS e
    ON n.entity_id = e.id
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC NULLS LAST
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,World,OWID_WRL,20192.37,14075.5080,24744.6580,605771.1110
1,Lower Middle Income (WB),None,15193.96,10202.5280,19182.7950,455818.7815
2,South Asia (WB),None,11729.89,7271.2754,15361.8780,351896.6626
3,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.9580,342729.8068
4,G20,None,10189.05,6339.0796,13164.8810,305671.3801
5,India,IND,8599.17,5425.8726,11121.1420,257975.1984
6,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.9770,142448.0427
7,African Region (WHO),None,4649.70,4283.0730,4917.7840,139491.0713
8,Low Income (WB),None,3568.11,3014.6170,3873.5330,107043.1635
9,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


Завдання 4

In [16]:
query = """
SELECT
    year,
    MAKE_DATE(year, 1, 1) AS first_january,
    CURRENT_DATE AS current_date,
    EXTRACT(
        YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1))
    )::INTEGER AS year_difference
FROM pandemic.infectious_cases_normalized
ORDER BY year DESC
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,year,first_january,current_date,year_difference
0,2023,2023-01-01,2026-10-16,3
1,2022,2022-01-01,2026-10-16,4
2,2022,2022-01-01,2026-10-16,4
3,2022,2022-01-01,2026-10-16,4
4,2022,2022-01-01,2026-10-16,4
5,2022,2022-01-01,2026-10-16,4
6,2022,2022-01-01,2026-10-16,4
7,2022,2022-01-01,2026-10-16,4
8,2022,2022-01-01,2026-10-16,4
9,2022,2022-01-01,2026-10-16,4


Завдання 5

різниці в роках

In [17]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION pandemic.year_difference(p_year INTEGER)
        RETURNS INTEGER
        LANGUAGE SQL
        STABLE
        AS $$
            SELECT EXTRACT(
                YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1))
            )::INTEGER;
        $$;
    """))

print("year_difference")

year_difference


In [18]:
query = """
SELECT
    year,
    MAKE_DATE(year, 1, 1) AS first_january,
    CURRENT_DATE AS current_date,
    pandemic.year_difference(year) AS year_difference
FROM pandemic.infectious_cases_normalized
ORDER BY year DESC
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,year,first_january,current_date,year_difference
0,2023,2023-01-01,2026-10-16,3
1,2022,2022-01-01,2026-10-16,4
2,2022,2022-01-01,2026-10-16,4
3,2022,2022-01-01,2026-10-16,4
4,2022,2022-01-01,2026-10-16,4
5,2022,2022-01-01,2026-10-16,4
6,2022,2022-01-01,2026-10-16,4
7,2022,2022-01-01,2026-10-16,4
8,2022,2022-01-01,2026-10-16,4
9,2022,2022-01-01,2026-10-16,4


In [19]:
query = """
SELECT pandemic.year_difference(1996) AS year_difference;
"""

display(pd.read_sql(query, engine))

,year_difference
0,30


функція середньої кількості захворювань

In [20]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION pandemic.average_cases(
            p_cases NUMERIC,
            p_divisor INTEGER
        )
        RETURNS NUMERIC
        LANGUAGE SQL
        IMMUTABLE
        AS $$
            SELECT ROUND(p_cases / NULLIF(p_divisor, 0), 2);
        $$;
    """))

print("average_cases")

average_cases


In [21]:
query = """
SELECT
    e.entity,
    n.year,
    n.number_rabies,
    pandemic.average_cases(n.number_rabies::numeric, 12) AS avg_monthly,
    pandemic.average_cases(n.number_rabies::numeric, 4) AS avg_quarterly,
    pandemic.average_cases(n.number_rabies::numeric, 2) AS avg_halfyear
FROM pandemic.infectious_cases_normalized AS n
JOIN pandemic.entities AS e
    ON n.entity_id = e.id
WHERE n.number_rabies IS NOT NULL
ORDER BY n.year DESC
LIMIT 10;
"""

display(pd.read_sql(query, engine))

,entity,year,number_rabies,avg_monthly,avg_quarterly,avg_halfyear
0,Andorra,2019,0.000138,0.00,0.00,0.00
1,Argentina,2019,0.567866,0.05,0.14,0.28
2,American Samoa,2019,0.007448,0.00,0.00,0.00
3,African Region (WHO),2019,4378.748000,364.90,1094.69,2189.37
4,Angola,2019,7.964379,0.66,1.99,3.98
5,Antigua and Barbuda,2019,0.004845,0.00,0.00,0.00
6,Algeria,2019,1.346161,0.11,0.34,0.67
7,Albania,2019,0.014225,0.00,0.00,0.01
8,Afghanistan,2019,3.387213,0.28,0.85,1.69
9,Armenia,2019,0.369019,0.03,0.09,0.18
